# 🎧 Project 06: Audio Feature Analysis & Song Skip Propensity
### Music Streaming Personalization, Acoustic Profiling & Cost-Sensitive Classification

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟢 Beginner  
**Domain:** Media & Entertainment Analytics  

---
### Notebook Outline:
1. **Environment Configuration**
2. **Audio Track Ingestion & Acoustic Feature Inspection**
3. **Exploratory Data Analysis: Acoustic Fingerprints of Skipped Songs**
4. **Contextual Analysis (Curated Playlist vs. Algorithmic Radio)**
5. **Class Imbalance Mitigation: Class-Weighted Ensembles vs. SMOTE**
6. **Model Benchmarking: PR-AUC & Brier Score Diagnostics**
7. **Acoustic Feature Attribution & Streaming Personalization Rules**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score,
    brier_score_loss, classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Audio analytics environment initialized.")

In [ ]:
# Data Ingestion
df = pd.read_csv("data/audio_streaming_skips.csv")
print(f"Streaming Sessions: {len(df)}")
print(f"Overall Skip Rate: {df['skipped_within_30s'].mean():.2%}")
display(df.head(4))

In [ ]:
# Acoustic Fingerprint of Skipped vs Completed Tracks
acoustic_cols = ['danceability', 'energy', 'speechiness', 'acousticness', 'valence']

df_melted = df.melt(
    id_vars=['skipped_within_30s'], 
    value_vars=acoustic_cols, 
    var_name='Acoustic_Feature', 
    value_name='Feature_Value'
)

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_melted, x='Acoustic_Feature', y='Feature_Value', hue='skipped_within_30s', palette='coolwarm')
plt.title("Acoustic Feature Comparison: Skipped (1) vs. Played Through (0)", fontweight='bold')
plt.show()

In [ ]:
# Preprocessing & Model Evaluation
features = ['danceability', 'energy', 'loudness_db', 'speechiness', 'acousticness', 'valence', 'tempo_bpm', 'playback_context']
X = df[features]
y = df['skipped_within_30s']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

num_cols = ['danceability', 'energy', 'loudness_db', 'speechiness', 'acousticness', 'valence', 'tempo_bpm']
cat_cols = ['playback_context']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ]
)

# Benchmark Models
pipe_lr = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(class_weight='balanced', random_state=42))])
pipe_rf = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=150, max_depth=7, class_weight='balanced', random_state=42))])

pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

probs_lr = pipe_lr.predict_proba(X_test)[:, 1]
probs_rf = pipe_rf.predict_proba(X_test)[:, 1]

results = [
    {
        "Model": "Logistic Regression (Balanced)",
        "ROC-AUC": round(roc_auc_score(y_test, probs_lr), 4),
        "PR-AUC": round(average_precision_score(y_test, probs_lr), 4),
        "Brier Score": round(brier_score_loss(y_test, probs_lr), 4)
    },
    {
        "Model": "Random Forest (Balanced)",
        "ROC-AUC": round(roc_auc_score(y_test, probs_rf), 4),
        "PR-AUC": round(average_precision_score(y_test, probs_rf), 4),
        "Brier Score": round(brier_score_loss(y_test, probs_rf), 4)
    }
]
display(pd.DataFrame(results))

In [ ]:
# Precision-Recall Curves for Skip Prediction
prec_lr, rec_lr, _ = precision_recall_curve(y_test, probs_lr)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, probs_rf)

plt.figure(figsize=(10, 5))
plt.plot(rec_lr, prec_lr, label=f"Logistic Regression (PR-AUC = {average_precision_score(y_test, probs_lr):.3f})")
plt.plot(rec_rf, prec_rf, label=f"Random Forest (PR-AUC = {average_precision_score(y_test, probs_rf):.3f})", color='green')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve on Track Skip Prediction", fontweight='bold')
plt.legend()
plt.show()